In [1]:
!pip install -q seqeval

In [2]:
!git clone https://github.com/VinAIResearch/PhoNER_COVID19.git

fatal: destination path 'PhoNER_COVID19' already exists and is not an empty directory.


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from seqeval.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [4]:
def read_conll_file(file_path):
    sentences = []
    words = []
    labels = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sentences.append((words, labels))
                    words = []
                    labels = []
            else:
                parts = line.split()
                word = parts[0]
                label = parts[-1]
                words.append(word)
                labels.append(label)

        if words:
            sentences.append((words, labels))

    return sentences

data_dir = "PhoNER_COVID19/data/word/"
train_data = read_conll_file(os.path.join(data_dir, "train_word.conll"))
dev_data = read_conll_file(os.path.join(data_dir, "dev_word.conll"))
test_data = read_conll_file(os.path.join(data_dir, "test_word.conll"))

print(f"Num sentences Train: {len(train_data)}")
print(f"Num sentences Dev: {len(dev_data)}")
print(f"Num sentences Test: {len(test_data)}")

Num sentences Train: 5027
Num sentences Dev: 2000
Num sentences Test: 3000


In [5]:
word_counts = Counter()
for words, _ in train_data:
    word_counts.update(words)

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.items():
    if count >= 1:
        vocab[word] = len(vocab)

label_set = set()
for _, labels in train_data:
    label_set.update(labels)

label2idx = {"<PAD>": 0}
for label in sorted(list(label_set)):
    label2idx[label] = len(label2idx)

idx2label = {v: k for k, v in label2idx.items()}

print(f"Vocab Size: {len(vocab)}")
print(f"Num Labels: {len(label2idx)}")
print("Labels:", list(label2idx.keys()))

Vocab Size: 5243
Num Labels: 21
Labels: ['<PAD>', 'B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']


In [6]:
class NERDataset(Dataset):
    def __init__(self, data, vocab, label2idx):
        self.data = data
        self.vocab = vocab
        self.label2idx = label2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        words, labels = self.data[idx]

        word_ids = [self.vocab.get(w, self.vocab["<UNK>"]) for w in words]

        label_ids = [self.label2idx.get(l, self.label2idx["<PAD>"]) for l in labels]

        return torch.tensor(word_ids, dtype=torch.long), torch.tensor(label_ids, dtype=torch.long)

def collate_fn(batch):
    words, labels = zip(*batch)

    words_padded = pad_sequence(words, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)

    return words_padded, labels_padded

BATCH_SIZE = 32

train_dataset = NERDataset(train_data, vocab, label2idx)
dev_dataset = NERDataset(dev_data, vocab, label2idx)
test_dataset = NERDataset(test_data, vocab, label2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [7]:
class BiLSTMEncoderNER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes, dropout=0.3):
        super(BiLSTMEncoderNER, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        embedded = self.dropout(embedded)

        lstm_out, _ = self.bilstm(embedded)
        lstm_out = self.dropout(lstm_out)

        logits = self.fc(lstm_out)
        return logits

In [8]:
EMBEDDING_DIM = 200
HIDDEN_SIZE = 256
NUM_LAYERS = 5
NUM_CLASSES = len(label2idx)
EPOCHS = 10
LEARNING_RATE = 0.001

model = BiLSTMEncoderNER(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

BiLSTMEncoderNER(
  (embedding): Embedding(5243, 200, padding_idx=0)
  (bilstm): LSTM(200, 256, num_layers=5, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=21, bias=True)
)


In [9]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0

    for words, labels in dataloader:
        words = words.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(words)

        loss = criterion(logits.view(-1, NUM_CLASSES), labels.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, idx2label):
    model.eval()
    true_predictions = []
    true_labels = []

    with torch.no_grad():
        for words, labels in dataloader:
            words = words.to(device)
            logits = model(words)
            preds = torch.argmax(logits, dim=-1)

            for pred_seq, label_seq in zip(preds.cpu().numpy(), labels.numpy()):
                seq_preds = []
                seq_labels = []
                for p, l in zip(pred_seq, label_seq):
                    if l != 0:
                        seq_preds.append(idx2label[p])
                        seq_labels.append(idx2label[l])
                true_predictions.append(seq_preds)
                true_labels.append(seq_labels)

    f1 = f1_score(true_labels, true_predictions)
    return f1, true_labels, true_predictions

In [10]:
best_dev_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    dev_f1, _, _ = evaluate(model, dev_loader, idx2label)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Dev F1-Score: {dev_f1*100:.2f}%")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        torch.save(model.state_dict(), "best_bilstm_ner.pt")

print(f"\nSaved model with best F1-score on dev set: {best_dev_f1*100:.2f}%")

Epoch 01/10 | Train Loss: 0.9407 | Dev F1-Score: 29.34%
Epoch 02/10 | Train Loss: 0.3082 | Dev F1-Score: 75.54%
Epoch 03/10 | Train Loss: 0.1574 | Dev F1-Score: 82.61%
Epoch 04/10 | Train Loss: 0.1098 | Dev F1-Score: 84.12%
Epoch 05/10 | Train Loss: 0.0876 | Dev F1-Score: 86.60%
Epoch 06/10 | Train Loss: 0.0656 | Dev F1-Score: 88.62%
Epoch 07/10 | Train Loss: 0.0564 | Dev F1-Score: 87.62%
Epoch 08/10 | Train Loss: 0.0477 | Dev F1-Score: 86.87%
Epoch 09/10 | Train Loss: 0.0395 | Dev F1-Score: 88.45%
Epoch 10/10 | Train Loss: 0.0349 | Dev F1-Score: 89.41%

Saved model with best F1-score on dev set: 89.41%


In [11]:
model.load_state_dict(torch.load("best_bilstm_ner.pt"))

test_f1, test_targets, test_preds = evaluate(model, test_loader, idx2label)

print("="*60)
print(f"SUMMARY F1-SCORE ON TEST SET: {test_f1 * 100:.2f}%")
print("="*60)
print("\n--------Entity Classification Report--------")
print(classification_report(test_targets, test_preds))

SUMMARY F1-SCORE ON TEST SET: 88.54%

--------Entity Classification Report--------
                     precision    recall  f1-score   support

                AGE       0.89      0.96      0.92       582
               DATE       0.93      0.97      0.95      1654
             GENDER       0.90      0.94      0.92       462
                JOB       0.47      0.45      0.46       173
           LOCATION       0.87      0.92      0.89      4441
               NAME       0.71      0.59      0.64       318
       ORGANIZATION       0.81      0.80      0.81       771
         PATIENT_ID       0.92      0.96      0.94      2005
SYMPTOM_AND_DISEASE       0.84      0.77      0.81      1136
     TRANSPORTATION       0.82      0.87      0.85       193

          micro avg       0.87      0.90      0.89     11735
          macro avg       0.82      0.82      0.82     11735
       weighted avg       0.87      0.90      0.88     11735

